# GeoTwin Colab run + viewer

Run `bash colab_bootstrap.sh` once in this Colab runtime first. Then set the three paths below and run the cell. It runs the Gaussian command, waits for the automatic VGGT Trim001/Clean500 export, and asks whether to open the viewer. Choosing **Open viewer** embeds GeoTwin in this notebook and loads the GLB and SRT directly from the run folder. No download/re-upload or manually started server.

The viewer's inferred color fill is a view-only overlay; it does not modify the exported mesh.


In [ ]:
from pathlib import Path
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread
import shutil
import subprocess
import sys
from urllib.parse import urlencode

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from google.colab import output

# Edit only these input paths if your uploaded files are stored elsewhere.
PROJECT_DIR = Path("/content/sih2026-3d-recon-zoom-mesh-fix-v6").resolve()
VIDEO_PATH = Path("/content/DJI0004.MP4").resolve()
SRT_PATH = Path("/content/DJI0004.srt").resolve()
OUT_DIR = Path("outputs/srt_visual_odometry_run")  # relative to PROJECT_DIR

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_DIR}")
if not VIDEO_PATH.is_file() or not SRT_PATH.is_file():
    raise FileNotFoundError("Upload DJI0004.MP4 and DJI0004.srt to /content, or update VIDEO_PATH/SRT_PATH above.")

command = [
    sys.executable, "layer3_gpu/python/train_gpu.py",
    "--source", "real",
    "--video", str(VIDEO_PATH),
    "--log", str(SRT_PATH),
    "--assumed_gimbal_pitch_deg", "-45",
    "--camera_hfov_deg", "84",
    "--target_fps", "1.5",
    "--max_frames", "120",
    "--max_focal_drift_frac", "0.10",
    "--pose_source", "visual_odometry",
    "--vo_min_inliers", "20",
    "--vo_max_frame_gap", "3",
    "--keep_fraction", "1.0",
    "--depth_min", "30",
    "--depth_max", "250",
    "--n_depths", "128",
    "--depth_boundary_reject",
    "--max_iterations", "8000",
    "--densify_until_iter", "6500",
    "--max_gaussians", "200000",
    "--auto_vggt_mesh",
    "--out_dir", str(OUT_DIR),
]

display(Markdown("### Running Gaussian reconstruction, then VGGT + Trim001/Clean500…"))
result = subprocess.run(command, cwd=PROJECT_DIR)
if result.returncode != 0:
    raise RuntimeError(f"Reconstruction ended with exit code {result.returncode}. See the output above and vggt_postprocess_status.json.")

run_dir = (PROJECT_DIR / OUT_DIR).resolve()
mesh_path = run_dir / "vggt_trim001_clean500" / "vggt_poisson_depth12.glb"
if not mesh_path.is_file():
    raise FileNotFoundError(f"Training finished but expected GLB was not found: {mesh_path}")

# Put a copy of the matching SRT beside this run so the browser can fetch both
# assets from the same Colab-served project root without a second upload.
viewer_assets = run_dir / "geotwin_viewer_assets"
viewer_assets.mkdir(parents=True, exist_ok=True)
served_log = viewer_assets / SRT_PATH.name
shutil.copy2(SRT_PATH, served_log)
model_rel = mesh_path.relative_to(PROJECT_DIR).as_posix()
log_rel = served_log.relative_to(PROJECT_DIR).as_posix()
# The page is served from /web/, so these ../ paths resolve inside the same project server.
query = urlencode({"model": "../" + model_rel, "log": "../" + log_rel})
viewer_page = "web/geotwin_mvp.html?" + query

prompt = widgets.Output()
open_button = widgets.Button(description="Open GeoTwin viewer", button_style="success", icon="external-link")
skip_button = widgets.Button(description="Skip viewer", button_style="")
controls = widgets.HBox([open_button, skip_button])
server_info = {"server": globals().get("_GEOTWIN_HTTP_SERVER")}

def open_viewer(_):
    open_button.disabled = True
    skip_button.disabled = True
    with prompt:
        clear_output(wait=True)
        server = server_info["server"]
        if server is None:
            handler = partial(SimpleHTTPRequestHandler, directory=str(PROJECT_DIR))
            server = ThreadingHTTPServer(("0.0.0.0", 0), handler)
            Thread(target=server.serve_forever, daemon=True).start()
            server_info["server"] = server
            globals()["_GEOTWIN_HTTP_SERVER"] = server
        display(Markdown("Viewer is loading the run's GLB and matching DJI SRT directly from Colab."))
        output.serve_kernel_port_as_iframe(server.server_address[1], path=viewer_page, width="100%", height="850")

def skip_viewer(_):
    open_button.disabled = True
    skip_button.disabled = True
    with prompt:
        clear_output(wait=True)
        display(Markdown("Viewer skipped. Your GLB is saved at `" + str(mesh_path) + "`."))

open_button.on_click(open_viewer)
skip_button.on_click(skip_viewer)
display(Markdown("### Reconstruction and export finished. Open the interactive viewer now?"))
display(controls, prompt)
